# Wilcoxon signed-rank: fusion vs image-only (one-sided)

In [ ]:
import json
import os
import re
import statistics
from pathlib import Path

_p = Path.cwd().resolve()
for _ in range(12):
    if (_p / "experiments").is_dir() and (_p / "data").is_dir():
        PROJECT_ROOT = _p
        break
    _p = _p.parent
else:
    PROJECT_ROOT = Path.cwd().resolve()
os.chdir(PROJECT_ROOT)

from scipy.stats import wilcoxon


def load_test_macro_f1(exp_name: str) -> dict[int, float]:
    out: dict[int, float] = {}
    base = PROJECT_ROOT / "experiments" / exp_name / "metrics" / "experiments"
    for path in base.glob("seed_*_results.json"):
        m = re.match(r"seed_(\d+)_results\.json$", path.name)
        if not m:
            continue
        obj = json.loads(path.read_text(encoding="utf-8"))
        out[int(m.group(1))] = float(obj["test_metrics"]["test_macro_f1"])
    return out


fusion = load_test_macro_f1("phase3_robustness")
image = load_test_macro_f1("imageonly_robustness")
seeds = sorted(fusion.keys() & image.keys())
if fusion.keys() != image.keys():
    raise ValueError(f"seed mismatch: only fusion {sorted(set(fusion)-set(image))} only image {sorted(set(image)-set(fusion))}")

d = [fusion[s] - image[s] for s in seeds]
res = wilcoxon(d, alternative="greater", zero_method="wilcox")

alpha = 0.05
print("n", len(d))
print("median_diff", statistics.median(d))
print("statistic", res.statistic)
print("pvalue", res.pvalue)
print("reject_H0_fusion_gt_image", res.pvalue < alpha)